In [1]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: ഘടനാപരമായ ഔട്ട്പുട്ടുകൾക്കായി Pydantic മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരിച്ചുപിടിക്കേണ്ട **സ്കീമ** നിർവചിക്കുന്നു. Pydantic ഉപയോഗിച്ച് `response_format` ഉപയോഗിക്കുന്നത് ഉറപ്പാക്കുന്നു:
- ✅ ടൈപ്പ്-സുരക്ഷിതമായ ഡാറ്റ എക്സ്ട്രാക്ഷൻ
- ✅ സ്വയമേവ സാധുത ഉറപ്പാക്കൽ
- ✅ ഫ്രീ-ടെക്സ്റ്റ് പ്രതികരണങ്ങളിൽ നിന്ന് പാഴ്‌സിംഗ് പിശകുകൾ ഉണ്ടാകാതിരിക്കുക
- ✅ ഫീൽഡുകളുടെ അടിസ്ഥാനത്തിൽ എളുപ്പമുള്ള വ്യവസ്ഥാപരമായ റൂട്ടിംഗ്


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

റൂമുകൾ ലഭ്യമാണെന്നും പരിശോധിക്കാൻ **availability_agent** ഇത് വിളിക്കും. Python ഫംക്ഷൻ ഒരു AI-കോളബിൾ ടൂൾ ആയി മാറാൻ `@ai_function` ഡെക്കറേറ്റർ നമ്മൾ ഉപയോഗിക്കുന്നു:
- LLM-യ്ക്ക് JSON സ്കീമ സ്വയം生成് ചെയ്യാൻ
- പാരാമീറ്റർ പരിശോധന കൈകാര്യം ചെയ്യാൻ
- ഏജന്റുകൾക്ക് ഓട്ടോമാറ്റിക് വിളക്കം സാധ്യമാക്കാൻ

ഈ ഡെമോയ്ക്ക്:
- **സ്റ്റോക്ക്‌ഹോം, സിയാറ്റിൽ, ടോക്ക്യോ, ലണ്ടൻ, ആംസ്ടർഡാം** → റൂമുകൾ ഡെൽ✅
- **മറ്റ് എല്ലാ നഗരങ്ങളും** → റൂമുകൾ ഇല്ല ❌


In [3]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ഘട്ടം 3: റൂട്ടിംഗിനുള്ള കൺഡിഷൻ ഫംഗ്ഷനുകൾ നിർവചിക്കുക

ഈ ഫംഗ്ഷനുകൾ ഏജന്റിന്റെ പ്രതികരണം പരിശോധിച്ച് വർക്ക്‌ഫ്ലോയിൽ ഏതു പാത പിന്തുടരണമെന്ന് നിർണയിക്കുന്നു.

**പ്രധാന പാറ്റേണ്‍:**
1. സന്ദേശം `AgentExecutorResponse` ആണോ എന്ന് പരിശോധിക്കുക
2. ഘടനാപരമായ ഔട്ട്പുട്ട് (Pydantic മോഡൽ) പാഴ് ചെയ്യുക
3. റൂട്ടിംഗ് നിയന്ത്രിക്കാൻ `True` അല്ലെങ്കിൽ `False` മടങ്ങി നൽകുക

വർക്ക്ഫ്ലോ ഈ കൺഡിഷനുകൾ **എഡ്ജുകൾ**-ൽ വിലയിരുത്തി അടുത്ത് ഏത് എക്സിക്യൂട്ടർ വിളിക്കണമെന്ന് തീരുമാനിക്കും.


In [4]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## പടി 4: ഇഷ്‌ടാനുസൃത ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

എക്സിക്യൂട്ടറുകൾ ട്രാൻസ്ഫർമേഷനുകൾ അല്ലെങ്കിൽ സൈഡ് എഫാക്റ്റുകൾ നിർവഹിക്കുന്ന വേർക്ക്‌ഫ്ലോ ഘടകങ്ങളാണ്. അവസാന ഫലം പ്രദർശിപ്പിക്കുന്ന ഇഷ്‌ടാനുസൃത എക്സിക്യൂട്ടർ സൃഷ്ടിക്കാൻ ഞങ്ങൾ `@executor` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നു.

**പ്രധാന ആശയങ്ങൾ:**
- `@executor(id="...")` - ഒരു ഫങ്ഷൻ വേർക്ക്‌ഫ്ലോ എക്സിക്യൂട്ടറായി റെജിസ്റ്റർ ചെയ്യുന്നു
- `WorkflowContext[Never, str]` - ഇൻപുട്ട്/ഔട്ട്പുട്ടിനുള്ള തരം സൂചനകൾ
- `ctx.yield_output(...)` - അവസാന വേർക്ക്‌ഫ്ലോ ഫലം യീൽഡ് ചെയ്യുന്നു


In [5]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## പടി 5: പരിസ്ഥിതി ചാരങ്ങള്‍ ലോഡ് ചെയ്യുക

LLM ക്ലയന്റിനെ ക്രമീകരിക്കുക. ഈ ഉദാഹരണം следующих εργασιών ചെയ്യുന്നു:
- **GitHub മോഡലുകൾ** (GitHub ടോക്കൺ ഉപയോഗിച്ച് ഫ്രീ ടിയർ)
- **Azure OpenAI**
- **OpenAI**


In [6]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")

## ഘട്ടം 6: സ്‌ട്രക്ചേഡ് ഔട്ട്പുട്ടുകളോടുള്ള AI ഏജന്റുകൾ സൃഷ്ടിക്കുക

ഞങ്ങൾ `AgentExecutor` എന്നതിൽ പൊതിഞ്ഞ **മൂന്ന് വിദഗ്ധ ഏജന്റുകൾ** സൃഷ്ടിക്കുന്നു:

1. **availability_agent** - ടൂൾ ഉപയോഗിച്ച് ഹോട്ടൽ ലഭ്യത പരിശോധിക്കുന്നു  
2. **alternative_agent** - (කාമറ ലഭ്യമല്ലാതിരിക്കുമ്പോൾ) പകരം നഗരങ്ങൾ നിർദ്ദേശിക്കുന്നു  
3. **booking_agent** - (കമറകൾ ലഭ്യമായിരിക്കുമ്പോൾ) ബുക്കിംഗ് പ്രോത്സാഹിപ്പിക്കുന്നു

**പ്രധാന സവിശേഷതകൾ:**  
- `tools=[hotel_booking]` - ഏജന്റിന് ടൂൾ നൽകുന്നു  
- `response_format=PydanticModel` - ഘടനാപരമായ JSON ഔട്ട്‌പുട്ട് നിർബന്ധിക്കുന്നു  
- `AgentExecutor(..., id="...")` - വർക്‌ഫ്ലോ ഉപയോഗത്തിനായി ഏജന്റിനെ പൊതിക്കും


In [7]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 7: സവിശേഷ നിബന്ധനകളോടുകൂടിയ എഡ്ജുകളും Workflow നിർമ്മിക്കല്‍

ഇപ്പോൾ ഞങ്ങൾ `WorkflowBuilder` ഉപയോഗിച്ച് സവിശേഷ നിബന്ധനകളുള്ള റൂട്ടിംഗ് ഉപയോഗിച്ച് ഗ്രാഫ് നിർമ്മിക്കുന്നു:

**Workflow ഘടനം:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**പ്രധാന മേത്തഡുകൾ:**
- `.set_start_executor(...)` - പ്രവേശന ബിന്ദു സജ്ജീകരിക്കുന്നു
- `.add_edge(from, to, condition=...)` - സവിശേഷ നിബന്ധനകളോടെ എഡ്ജ് ചേർക്കുന്നു
- `.build()` - Workflow അന്തിമരൂപം നൽകുന്നു


In [8]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ഘട്ടം 8: ടെസ്റ്റ് കെയ്സ് 1 ഓടിക്കുക - ലഭ്യത ഇല്ലാത്ത നഗരം (പാരീസ്)

**ലഭ്യത ഇല്ലാത്ത** വഴിക്കായി പാരീസിലുള്ള ഹോട്ടലുകൾ ചോദിച്ച് പരിശോധിക്കാം (നമുടെ സിമുലേഷനിൽ അവിടെ ഒരു റൂമും ഇല്ല).


In [9]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: ടെസ്റ്റ് കേസ് 2 ഓടിക്കുക - ലഭ്യതയുള്ള നഗരം (സ്റ്റോക്ക്ഹോൾം)

ഇപ്പോൾ നമ്മുടെ സിമുലേഷനിൽ മുറികൾ ഉള്ള സ്റ്റോക്ക്ഹോൾമിലെ ഹോട്ടലുകൾ അഭ്യർത്ഥിച്ച് **ലഭ്യത** പാത പരീക്ഷിക്കാം.


In [10]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## പ്രധാന കാര്യങ്ങളും आगാം कदमുകളും

### ✅ നിങ്ങൾ അറിയുകൊണ്ടിരിക്കുന്നു:

1. **WorkflowBuilder മാതൃക**
   - പ്രവേശനകേന്ദ്രം നിർവചിക്കാൻ `.set_start_executor()` ഉപയോഗിക്കുക
   - ശർത്തികൾ അടിസ്ഥാനമാക്കിയുള്ള മാർഗ്ഗനിർദ്ദേശത്തിനായി `.add_edge(from, to, condition=...)` ഉപയോഗിക്കുക
   - വർക്ക്‌ഫ്ലേ അവസാനിപ്പിക്കാൻ `.build()` വിളിക്കുക

2. **ശർത്തി അടിസ്ഥാനമാക്കിയുള്ള മാർഗ്ഗനിർദ്ദേശം**
   - അവസ്ഥ ഫങ്ഷനുകൾ `AgentExecutorResponse` പരിശോധിക്കുന്നു
   - മാർഗ്ഗനിർദ്ദേശ തീരുമാനം എടുക്കാൻ ഘടിത ഔട്പുട്ട് പാർസ് ചെയ്യുക
   - ഒരു എഡ് സജീവമാക്കാൻ `True` തിരികെ നൽകുക, ഒഴിവാക്കാൻ `False`

3. **ടൂൾ ഐന്റഗ്രേഷൻ**
   - പൈത്തൺ ഫങ്ഷനുകൾ AI ടൂളുകളാക്കി മാറ്റാൻ `@ai_function` ഉപയോഗിക്കുക
   - ഏജന്റുകൾ ആവശ്യമായപ്പോൾ ടൂളുകൾ സ Fédération
   - ടൂളുകൾ JSON തരികയും ഏജന്റുകൾ Parse ചെയ്യുകയും ചെയ്യാം

4. **ഘടിത ഔട്പുട്ടുകൾ**
   - ടൈപ്പ് സെഫായി ഡാറ്റ എക്സ്ട്രാക്ഷനായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക
   - ഏജന്റുകൾ സൃഷ്ടിക്കുമ്പോൾ `response_format=MyModel` സജ്ജമാക്കുക
   - മറുപടികൾ `Model.model_validate_json()` ഉപയോഗിച്ച് പാർസ് ചെയ്യുക

5. **കസ്റ്റം എക്സിക്യൂട്ടറുകൾ**
   - വർക്‌ഫ്ലേ ഘടകങ്ങൾ സൃഷ്ടിക്കാൻ `@executor(id="...")` ഉപയോഗിക്കുക
   - ഡാറ്റ രൂപാന്തരം ചെയ്യാനോ മറ്റു സൈഡ് ഇഫക്റ്റുകൾ നടത്താനോ എക്സിക്യൂട്ടറുകൾ കഴിയും
   - വർക്‌ഫ്ലേ ഫലങ്ങൾ ഉത്പാദിപ്പിക്കാൻ `ctx.yield_output()` ഉപയോഗിക്കുക

### 🚀 യാഥാർത്ഥ്യ പ്രയോഗങ്ങൾ:

- **ട്രാവൽ ബുക്കിംഗ്**: ലഭ്യത പരിശോധിക്കുക, ബകലും ഉപയോക്തൃ നിർദ്ദേശങ്ങൾ നൽകുക, ഓപ്ഷനുകൾ താരതമ്യം ചെയ്യുക
- **കസ്റ്റമർ സർവീസ്**: പ്രശ്ന തരം, വികാരം, പ്രാധാന്യം പരിഗണിച്ച് റൂട്ടിംഗ്
- **ഇ-കോമേഴ്‌സ്**: ഇന്വെൻട്രി പരിശോധിക്കുക, ബകലും ഉപയോക്തൃ നിർദ്ദേശങ്ങൾ നൽകുക, ഓർഡറുകൾ പ്രോസസ്സ് ചെയ്യുക
- **കന്റന്റ് മോഡറേഷൻ**: വിഷമത സ്കോർ, ഉപയോക്തൃ ഫ്‌ലാഗുകൾ അനുസരിച്ച് റൂട്ടിംഗ്
- **അപ്ഫ്രൂവൽ വർക്‌ഫ്ലോകൾ**: തുക, ഉപയോക്തൃ പദവി, അപകട നില അനുസരിച്ച് റൂട്ടിംഗ്
- **പല ഘട്ട പ്രോസസ്സ്**: ഡാറ്റ ഗുണനിലവാരം, പൂര്‍ണ്ണത പരിഗണിച്ച് റൂട്ടിംഗ്

### 📚 അടുത്ത ചുവടുകൾ:

- കൂടുതൽ സങ്കീർണമായ അവസ്ഥകൾ (അനേകം മാനദണ്ഡങ്ങൾ) ചേർക്കുക
- വർക്‌ഫ്ലേ സ്റ്റേറ്റ് മാനേജ്മെന്റോടൊപ്പം ലൂപ്പുകൾ നടപ്പിലാക്കുക
- പുനരുപയോഗയോഗ്യ ഘടകങ്ങൾക്കായി ഉപ-വർക്‌ഫ്ലോകൾ ചേർക്കുക
- യഥാർത്ഥ API-കളുമായി ഐന്റഗ്രേറ്റ് ചെയ്യുക (ഹോട്ടൽ ബുക്കിംഗ്, ഇൻവെൻട്രി സിസ്റ്റങ്ങൾ)
- പിശക് കൈമാറ്റവും ബാക്കപ്പ് പാതകളും ചേർക്കുക
- ഉൾപ്പെടുത്തിയിരിക്കുന്ന ദൃശ്യവൽക്കരണ ഉപകരണങ്ങളോടെ വർക്‌ഫ്ലോകൾ ദൃശ്യവൽക്കരിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**തെറ്റിരിപ്പ്**:  
ഈ പ്രമാണം AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിച്ചിട്ടുണ്ടെങ്കിലും, യന്ത്ര വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ ശരിയായില്ലാത്തതൊക്കേ ഇരു വഴി സംഭവിക്കാവുന്നതാണ്. മാതൃഭാഷയിലെ മുള്‍ പ്രമാണം ഔദ്യോഗികമായ ഉറവിടമായി കാണാൻ നിർദേശിക്കുന്നു. തീർത്ഥചിഹ്നമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനവ വിവർത്തനം അഭ്യർത്ഥിക്കുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിൽ നിന്നുള്ള ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കോ വ്യാഖ്യാന വഴിപ്പിഴവുകൾക്കോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
